In [11]:
import pandas as pd


match = pd.read_csv('./data/gold_match.csv')


df_ohl = match.copy()

# Order by date
df_ohl = df_ohl.sort_values(by='match_date').reset_index(drop=True)

# victory 3 points, draw 1, defeat 0

def ohl_points(row):
    # if ohl played home
    if row['home_team'] == 'OH Leuven':
        if row['result_home'] == 'W': return 3
        elif row['result_home'] == 'D': return 1
        else: return 0
    # if played out
    else:
        if row['result_home'] == 'L': return 3
        elif row['result_home'] == 'D': return 1
        else: return 0

df_ohl['ohl_match_points'] = df_ohl.apply(ohl_points, axis=1)

#calculate the 3 previous game, without the first one
df_ohl['ohl_recent_form_points'] = df_ohl['ohl_match_points'].rolling(window=3).sum().shift(1)

# only home games
df_home_form = df_ohl[df_ohl['is_home_match'] == True][['match_id', 'match_date', 'away_team', 'tickets_scanned', 'ohl_recent_form_points']].copy()

# ignore the first games 
df_home_form['ohl_recent_form_points'] = df_home_form['ohl_recent_form_points'].fillna(4)

print("Sucess!")
print(df_home_form.head(10))

Sucess!
                     match_id  match_date             away_team  \
1   d256yo3eng04m0fu7b4sl7wno  2022-07-30              Westerlo   
3   d4mn5ksbxuvnaww4pmommxhqs  2022-08-14           Club Brugge   
5   d65hmi7sq03yzr5he1k7ypus4  2022-08-27           KV Oostende   
7   d80mkemezkz16bqh6lbn8tlhw  2022-09-10    Sporting Charleroi   
9   dak40etbhbqsr1nxyt50qcg0k  2022-10-01  Union Saint-Gilloise   
11  dcl81t8chgq07o8rw8orj5no4  2022-10-15                  Genk   
14  dfui3aovgvfccasonp6w5ajh0  2022-10-30                  Gent   
16  di0x0as1g324k0hoea1o5vvh0  2022-11-11               Seraing   
18  dkbsomyigazbberxvdmjq7wgk  2023-01-08              Kortrijk   
20  dmcif0n7veawzigj8cd5bm904  2023-01-17                 Eupen   

    tickets_scanned  ohl_recent_form_points  
1            5565.0                     4.0  
3            7440.0                     6.0  
5            4489.0                     3.0  
7            4508.0                     7.0  
9            6290.0     

In [12]:
# sort by recent points

# ascending false biiger to smaller
df_ordenado_pontos = df_home_form.sort_values(by='ohl_recent_form_points', ascending=False)

print("\n------ Top games by previous points -----")
print(df_ordenado_pontos[['match_date', 'away_team', 'ohl_recent_form_points', 'tickets_scanned']].head(20))


------ Top games by previous points -----
     match_date             away_team  ohl_recent_form_points  tickets_scanned
7    2022-09-10    Sporting Charleroi                     7.0           4508.0
35   2023-08-05                  RWDM                     7.0           7786.0
128  2025-11-23          Sint-Truiden                     7.0           5661.0
73   2024-05-25              Westerlo                     7.0           5907.0
42   2023-09-30        Standard Liège                     7.0           9211.0
29   2023-03-19            Anderlecht                     7.0          10827.0
3    2022-08-14           Club Brugge                     6.0           7440.0
121  2025-09-20      RAAL La Louvière                     6.0           4036.0
59   2024-02-17    Sporting Charleroi                     6.0           9228.0
33   2023-04-23        Standard Liège                     6.0          10079.0
138  2026-02-14                Dender                     5.0           5137.0
9    2022

In [13]:

# order by most sold tickets

df_ordenado_bilhetes = df_home_form.sort_values(by='tickets_scanned', ascending=False)

print("\n------- Top games: most tickets sold ---------")
print(df_ordenado_bilhetes[['match_date', 'away_team', 'tickets_scanned', 'ohl_recent_form_points']].head(15))


------- Top games: most tickets sold ---------
    match_date             away_team  tickets_scanned  ohl_recent_form_points
63  2024-03-17              Mechelen          11069.0                     1.0
29  2023-03-19            Anderlecht          10827.0                     7.0
54  2024-01-21            Anderlecht          10786.0                     3.0
89  2024-12-01            Anderlecht          10723.0                     2.0
48  2023-11-26           Club Brugge          10546.0                     0.0
86  2024-11-02           Club Brugge          10485.0                     4.0
33  2023-04-23        Standard Liège          10079.0                     6.0
61  2024-03-02  Union Saint-Gilloise           9789.0                     2.0
40  2023-09-17                  Gent           9598.0                     4.0
26  2023-02-26               Antwerp           9331.0                     1.0
46  2023-11-04              Westerlo           9240.0                     3.0
59  2024-02-17  

In [14]:
# Linear regression

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# clean data
df_model = df_home_form.dropna(subset=['tickets_scanned']).copy()

# Convert away_team text to binary
X = pd.get_dummies(df_model[['away_team', 'ohl_recent_form_points']], drop_first=True)

# Target -> tickets scanned
y = df_model['tickets_scanned']

# Split data: 80% train 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# train
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

# predictions 
previsoes = model_lr.predict(X_test)


mae = mean_absolute_error(y_test, previsoes)
r2 = r2_score(y_test, previsoes)

print("\n------- Results -------")
print(f"MAE (Erro Médio Absoluto): Erramos por ~{int(mae)} bilhetes em média.")

# r2 indicates variance explained by model
print(f"R² Score: {r2:.2f} (O modelo explica {int(r2*100)}% da variação na assistência)") 


------- Results -------
MAE (Erro Médio Absoluto): Erramos por ~2053 bilhetes em média.
R² Score: -0.70 (O modelo explica -70% da variação na assistência)
